# Voltametria Reversível

## Bibliotecas Necessárias:

In [ ]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
from scipy import special
from ipywidgets import Text, Button, HBox, VBox, Output
from IPython.display import display, clear_output

## Definindo o Modelo:

In [ ]:
model = pybamm.BaseModel()

co = pybamm.Variable("Concentração de O", domain="electrolyte")
cr = pybamm.Variable("Concentração de R", domain="electrolyte")

Ei = pybamm.Parameter("Potencial Inicial [V]")
Ef = pybamm.Parameter("Potencial Final [V]")
E0 = pybamm.Parameter("Potencial Padrão [V]")
v  = pybamm.Parameter("Velocidade de Varredura [V.s-1]")
F  = pybamm.Parameter("Constante de Faraday [C.mol-1]")
R  = pybamm.Parameter("Constante dos Gases [J.K-1.mol-1]")
T  = pybamm.Parameter("Temperatura [K]")

No = -pybamm.grad(co)   # fluxo difusional de O
Nr = -pybamm.grad(cr)   # fluxo difusional de R

model.rhs = {
    co: -pybamm.div(No),  # lei de Fick para O
    cr: -pybamm.div(Nr),  # lei de Fick para R
}

# condições iniciais
model.initial_conditions = {
    co: pybamm.Scalar(1),  # c_O(x,0) = 1: O uniforme e adimensionalizado
    cr: pybamm.Scalar(0),  # c_R(x,0) = 0: R inicialmente ausente
}

# condições de contorno — Dirichlet
# esquerda (x=0): interface eletrodo/solução — equilíbrio de Nernst, potencial varia com t
# direita  (x=6): seio da solução            — difusão semi-infinita
f    = F / (R * T)
E    = Ei + v * pybamm.t        # potencial aplicado varia linearmente com o tempo (varredura)
eta  = E - E0
teta = pybamm.exp(f * eta)

model.boundary_conditions = {
    co: {
        "left":  (teta / (1 + teta), "Dirichlet"),  # c_O(0,t) = θ/(1+θ) — equilíbrio de Nernst
        "right": (pybamm.Scalar(1),  "Dirichlet"),  # c_O(∞,t) = 1       — difusão semi-infinita
    },
    cr: {
        "left":  (1 / (1 + teta),   "Dirichlet"),  # c_R(0,t) = 1/(1+θ) — equilíbrio de Nernst
        "right": (pybamm.Scalar(0),  "Dirichlet"),  # c_R(∞,t) = 0       — difusão semi-infinita
    },
}

model.variables = {
    "Concentração de O": co,
    "Concentração de R": cr,
    "Fluxo de O": No,
    "Fluxo de R": Nr,
    "Potencial Aplicado": E,
}

param = pybamm.ParameterValues(
    {
        "Potencial Inicial [V]": "[input]",
        "Potencial Final [V]": "[input]",
        "Potencial Padrão [V]": "[input]",
        "Velocidade de Varredura [V.s-1]": "[input]",
        "Constante de Faraday [C.mol-1]": 96485.3,
        "Constante dos Gases [J.K-1.mol-1]": 8.31446,
        "Temperatura [K]": 298.15,
    }
)

# ── geometria e malha ────────────────────────────────────────────────────────
x_var = pybamm.SpatialVariable("x", domain=["electrolyte"], coord_sys="cartesian")

geometry = {"electrolyte": {x_var: {"min": pybamm.Scalar(0), "max": pybamm.Scalar(6)}}}

submesh_types = {"electrolyte": pybamm.Uniform1DSubMesh}
var_pts       = {x_var: 400}
mesh          = pybamm.Mesh(geometry, submesh_types, var_pts)

spatial_methods = {"electrolyte": pybamm.FiniteVolume()}
disc = pybamm.Discretisation(mesh, spatial_methods)

param.process_model(model)
param.process_geometry(geometry)
disc.process_model(model)

## Gráfico Estático:

In [ ]:
solver = pybamm.ScipySolver()

# potenciais fixos para a varredura (eq. de varredura linear de potencial)
EI_PADRAO = 0.2   # potencial inicial [V]
EF_PADRAO = -0.2  # potencial final [V]
E0_PADRAO = 0     # potencial padrão [V]

velocidades = [-0.02, -0.05, -0.1, -0.2, -0.5]  # velocidades de varredura [V/s]
cores       = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

correntes        = []
potenciais       = []
correntes_de_pico = []

for vi in velocidades:
    tf = (EF_PADRAO - EI_PADRAO) / vi
    t  = np.linspace(0, tf, 1000)

    solucao = solver.solve(
        model, t,
        inputs={
            "Potencial Inicial [V]": EI_PADRAO,
            "Potencial Final [V]": EF_PADRAO,
            "Potencial Padrão [V]": E0_PADRAO,
            "Velocidade de Varredura [V.s-1]": vi,
        }
    )

    No_sol = solucao["Fluxo de O"]
    E_sol  = solucao["Potencial Aplicado"]

    i = No_sol(solucao.t, x=0)  # corrente catódica negativa, sem inversão de sinal
    E = E_sol(solucao.t)

    correntes.append(i)
    potenciais.append(E)
    correntes_de_pico.append(np.min(i))  # pico de corrente catódica (mínimo, pois é negativa)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

for vi, cor, i, E in zip(velocidades, cores, correntes, potenciais):
    ax1.plot(E, i, color=cor, label=f"v = {vi}")

ax1.set_xlabel("E - E0 / V")
ax1.set_ylabel("Corrente")
ax1.set_xlim([EF_PADRAO, EI_PADRAO])  # eixo X: negativo à esquerda, positivo à direita
ax1.set_title("Voltamogramas para diferentes velocidades de varredura")
ax1.legend(loc="lower left")

raizes_v = [np.sqrt(abs(vi)) for vi in velocidades]
ax2.plot(raizes_v, correntes_de_pico, "o", color="tab:blue")
ax2.set_xlabel("Raiz da Velocidade de Varredura")
ax2.set_ylabel("Corrente de Pico")
ax2.set_title("Corrente de pico vs raiz da velocidade")

plt.tight_layout()
plt.show()

## Gráfico Interativo: 

In [ ]:
solver_int = pybamm.ScipySolver()

saida = Output()

def plotar(ei1_str, ef1_str, e01_str, v1_str, ei2_str, ef2_str, e02_str, v2_str):
    with saida:
        clear_output(wait=True)

        try:
            ei1 = float(ei1_str)
            ef1 = float(ef1_str)
            e01 = float(e01_str)
            v1  = float(v1_str)
            ei2 = float(ei2_str)
            ef2 = float(ef2_str)
            e02 = float(e02_str)
            v2  = float(v2_str)
        except ValueError:
            print("Por favor, insira valores numéricos válidos.")
            return

        if v1 == 0 or v2 == 0:
            print("A velocidade de varredura não pode ser zero.")
            return

        # ── curva 1 ──────────────────────────────────────────────────────────
        tf1  = (ef1 - ei1) / v1
        sol1 = solver_int.solve(
            model, np.linspace(0, tf1, 1000),
            inputs={
                "Potencial Inicial [V]": ei1,
                "Potencial Final [V]": ef1,
                "Potencial Padrão [V]": e01,
                "Velocidade de Varredura [V.s-1]": v1,
            }
        )
        i1 = sol1["Fluxo de O"](sol1.t, x=0)
        E1 = sol1["Potencial Aplicado"](sol1.t)

        # ── curva 2 ──────────────────────────────────────────────────────────
        tf2  = (ef2 - ei2) / v2
        sol2 = solver_int.solve(
            model, np.linspace(0, tf2, 1000),
            inputs={
                "Potencial Inicial [V]": ei2,
                "Potencial Final [V]": ef2,
                "Potencial Padrão [V]": e02,
                "Velocidade de Varredura [V.s-1]": v2,
            }
        )
        i2 = sol2["Fluxo de O"](sol2.t, x=0)
        E2 = sol2["Potencial Aplicado"](sol2.t)

        # ── plot ─────────────────────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(7, 4))

        ax.plot(E1, i1, color="tab:blue", linewidth=2, label="1ª Varredura")
        ax.plot(E2, i2, color="tab:red",  linewidth=2, label="2ª Varredura")

        ax.set_xlim([min(ei1, ef1, ei2, ef2), max(ei1, ef1, ei2, ef2)])
        ax.set_xlabel("E / V")
        ax.set_ylabel("Corrente")
        ax.set_title("Comparação de Voltamogramas")
        ax.legend()

        plt.tight_layout()
        display(fig)
        plt.close(fig)

# ── widgets — 1ª varredura ────────────────────────────────────────────────────
campo_ei1 = Text(value="0.5",  description="Potencial Inicial 1 [V]:", style={"description_width": "initial"})
campo_ef1 = Text(value="-0.5", description="Potencial Final 1 [V]:",   style={"description_width": "initial"})
campo_e01 = Text(value="0",    description="Potencial Padrão 1 [V]:",  style={"description_width": "initial"})
campo_v1  = Text(value="-0.1", description="Velocidade 1 [V/s]:",      style={"description_width": "initial"})

# ── widgets — 2ª varredura ────────────────────────────────────────────────────
campo_ei2 = Text(value="0.5",  description="Potencial Inicial 2 [V]:", style={"description_width": "initial"})
campo_ef2 = Text(value="-0.5", description="Potencial Final 2 [V]:",   style={"description_width": "initial"})
campo_e02 = Text(value="0",    description="Potencial Padrão 2 [V]:",  style={"description_width": "initial"})
campo_v2  = Text(value="-0.2", description="Velocidade 2 [V/s]:",      style={"description_width": "initial"})

botao = Button(description="Recalcular", button_style="success")

def ao_clicar(b):
    plotar(
        campo_ei1.value, campo_ef1.value, campo_e01.value, campo_v1.value,
        campo_ei2.value, campo_ef2.value, campo_e02.value, campo_v2.value,
    )

botao.on_click(ao_clicar)

interface = VBox([
    HBox([campo_ei1, campo_ef1, campo_e01, campo_v1]),
    HBox([campo_ei2, campo_ef2, campo_e02, campo_v2]),
    botao,
    saida,
])
display(interface)

# gráfico inicial
plotar(
    campo_ei1.value, campo_ef1.value, campo_e01.value, campo_v1.value,
    campo_ei2.value, campo_ef2.value, campo_e02.value, campo_v2.value,
)